In [1]:
%pip install scikit-learn -q
%pip install xgboost -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import seaborn as sns 

from xgboost import XGBRegressor

In [3]:
file_features_path = f'../data/feet_dataset_clean.csv'
file_labels_path = f'../data/feet_labels.csv'

feet_features_df = pd.read_csv(file_features_path)
feet_features_df = feet_features_df[ feet_features_df['bad_photo'] != 1 ]

feet_labels_df = pd.read_csv(file_labels_path)
feet_features_df = feet_features_df.copy()

# Limpia la extensión de los nombres
feet_features_df['img_name_clean'] = feet_features_df['img_name'].str.rsplit('.', n=1).str[0]

# Aplica la transformación adicional con lambda
feet_features_df['img_name_clean'] = feet_features_df['img_name_clean'].apply(
    lambda x: '_'.join(x.split('_')[:-1]) if x.split('_')[-1].isdigit() else x
)
full_feet_df = pd.merge(feet_features_df, feet_labels_df, how='left', left_on='img_name_clean', right_on='name')

sel_columns = ["width", "height", "coin_width", "coin_height", "foot_width", "sex", "foot_side", "size_cm"]
feet_df = full_feet_df[sel_columns]
feet_df

,width,height,coin_width,coin_height,foot_width,sex,foot_side,size_cm
0,1280,960,123,122,1061,M,left,27.2
1,1280,960,118,116,1032,M,left,27.2
2,1280,960,113,112,988,M,left,27.2
3,1280,960,110,108,1025,M,left,27.2
4,1280,960,109,107,1022,M,left,27.2
...,...,...,...,...,...,...,...,...
615,1600,1200,173,174,1412,F,left,23.9
616,1600,1200,146,149,1219,F,left,23.9
617,1600,1200,163,163,1333,F,left,23.9
618,1600,1200,146,149,1220,F,left,23.9


In [4]:

def collect_metrics(model_name, y_pred, y_test):
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    print(f"-------model: {model_name}-------")
    print(f"R²: {r2:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print()

feet_encoded = pd.get_dummies(feet_df, columns=['sex', 'foot_side'],  drop_first=True)

X = feet_encoded.drop(columns=['size_cm'])
y = feet_encoded["size_cm"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=32, shuffle=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42).fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
collect_metrics("XGBRegressor",y_pred_xgb, y_test)

-------model: XGBRegressor-------
R²: 0.8509
MSE: 0.4881
RMSE: 0.6987



In [7]:
residuals = y_test - y_pred_xgb

# Crear un DataFrame para organizar los datos
residuals_df = pd.DataFrame({
    'Ground Truth': y_test,
    'Prediction': y_pred_xgb,
    'Residual': residuals,
    'Abs Residual': np.abs(residuals)  # Residuales en valor absoluto
})

# Ordenar por los residuales absolutos en orden descendente
top_residuals = residuals_df.sort_values(by='Abs Residual', ascending=False).head(10)
top_residuals

,Ground Truth,Prediction,Residual,Abs Residual
242,25.5,23.352554,2.147446,2.147446
494,21.7,23.580530,-1.880530,1.880530
65,25.6,27.455709,-1.855709,1.855709
45,24.3,25.901920,-1.601920,1.601920
584,23.6,25.196743,-1.596743,1.596743
39,24.3,25.870474,-1.570474,1.570474
241,25.5,23.976667,1.523333,1.523333
572,23.6,25.121233,-1.521233,1.521233
37,24.3,25.792089,-1.492089,1.492089
44,24.3,25.792089,-1.492089,1.492089
